In [1]:
from pathlib import Path

from imagematerials.buildings import buildings_preprocessing
from imagematerials.vehicles.preprocessing import preprocessing as vehicles_preprocessing
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.__main__ import simulate_stocks
from imagematerials.factory2 import ModelFactory
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks, Maintenance, MaterialIntensities
from imagematerials.concepts import building_knowledge_graph

import prism


In [2]:
base_dir = Path("..", "IMAGE-Mat_old_version", "IMAGE-Mat", "BUMA")
prep_building_fp = Path("prep_buildings.nc")

if not prep_building_fp.is_file():
    prep_data_buildings = buildings_preprocessing(base_dir)
    export_to_netcdf(prep_data_buildings, prep_building_fp)
else:
    prep_data_buildings = import_from_netcdf(prep_building_fp)

In [3]:
base_dir = "../data/raw"
vehicle_prep_fp = Path("prep_vema2.nc")

if not vehicle_prep_fp.is_file():
    _, orig_prep_data = vehicles_preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, vehicle_prep_fp)
vehicle_prep_data = import_from_netcdf(vehicle_prep_fp)
vehicle_prep_data["weights"] = vehicle_prep_data.pop("vehicle_weights")

In [4]:
prep_data = {
    "buildings": prep_data_buildings,
    "vehicles": vehicle_prep_data,
}

In [5]:
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)

In [ ]:
factory = ModelFactory(prep_data, complete_timeline)
factory.add(GenericStocks, "buildings")
factory.add(GenericStocks, "vehicles")
factory.add(GenericMaterials, "buildings")
factory.add(GenericMaterials, "vehicles")
model = factory.finish()

In [10]:
simulation_timeline = prism.Timeline(1970, 2060, 1)
model.simulate(simulation_timeline)

In [13]:
model.buildings["stock_by_cohort"]

<xarray.DataArray (Time: 340, Cohort: 340, Region: 26, Type: 12)> Size: 289MB
array([[[[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
          0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
...
         [0.00000000e+00, 6.29099303e+00, 0.00000000e+00, ...,
          1.51892840e-01, 5.24297198e-02, 1.79858339e-01],
         [1.17389380e+01, 6.07633353e+01, 1.18842989e+02, ...,
          1.53576540e-02, 1.09940845e-02, 1.11986742e-01],
         [4.60420552e+00, 3.51190709e+01, 7.56244891e+01, ...,
          1.00187665e-02, 8.61599439e-03, 9.79890642e-02]],

        [[1.48315192e-01, 5.57017338e+00, 2.43609034e+00, ...,
          1.46270634e-01, 5.21959503e-02, 1.77391196e-01],
         [0.00000000e+00, 2.65606715e+01, 0.00000000e+00, ...,
          1.41355288e-01, 5.07142211e-02, 1.76080702e-01],
         [4.75715875e-01, 2.87860538e+01, 7.81367595e+00, ...,
          1.55481595e-01, 4.73042562e-02, 1.93458602e-01],
         ...,
         [0.00000000e+00, 6.26778045e+00, 0.00000000e+00, ...,
          1.54212370e-01, 5.35352037e-02, 1.79900161e-01],
         [1.18320150e+01, 6.23468297e+01, 1.19785286e+02, ...,
          1.65793509e-02, 1.15260434e-02, 1.15119615e-01],
         [4.73872338e+00, 3.67408424e+01, 7.78339569e+01, ...,
          1.09849736e-02, 9.14227596e-03, 1.01871618e-01]]]])
Coordinates:
  * Time     (Time) int64 3kB 1721 1722 1723 1724 1725 ... 2057 2058 2059 2060
  * Cohort   (Cohort) int64 3kB 1721 1722 1723 1724 1725 ... 2057 2058 2059 2060
  * Region   (Region) <U2 208B '1' '2' '3' '4' '5' ... '22' '23' '24' '25' '26'
  * Type     (Type) <U21 1kB 'Appartment - Rural' ... 'Govt+'